# 08 — Topology evolution with Meta Agent Search

**ADAS-style structural search over typed `TopologySpec` candidates.**

GEPA evolves *prompts* inside a fixed topology (notebooks 06 / 07). This notebook evolves the *topology itself* — a meta agent designs Pipeline / Parallel / Router compositions, evaluates each against a multi-bucket gold set, and keeps the top-K on a Pareto frontier of accuracy / cost / latency / `n_agents` / `depth`.

Faithful to ADAS ([arXiv 2408.08435](https://arxiv.org/abs/2408.08435)) in shape (design → reflexion → evaluate → archive). **Diverges in surface:** the meta agent emits typed `TopologySpec` JSON, not raw Python — no `exec()`, no sandbox, no code-execution failure modes. Expressivity is capped to compositions of registered agents and callables (the orqest equivalent of AFlow's operator vocabulary).

What you'll see:

1. A 3-bucket gold set (factual / inferential / abstain) the seed CoT topology underperforms on.
2. A small search budget (5 generations × 3 examples per minibatch ≈ \$1–2 in real LLM calls).
3. The discovered topology + the meta agent's design reasoning.
4. The Pareto frontier across 5 dimensions.
5. Per-bucket lift measured against the seed.


## 1. Setup — config + agents + registries

In [1]:
from __future__ import annotations

from typing import Any
from pydantic import BaseModel, Field

from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()
print(f"Task model:    {config.llm_model}")
print(f"Reflection:    {config.llm_model} (overridden via MetaAgentConfig if you want a stronger one)")


class Answer(BaseModel):
    answer: str = Field(description="Direct answer or 'the paragraph does not say'.")


class CotAgent(BaseAgent[GlobalState, Answer]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output


class VerifierAgent(BaseAgent[GlobalState, Answer]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(
            f"Re-check this answer for the question. If wrong, correct it. Question and prior answer:\n{latest}",
            state,
        )
        return result.output


def make_cot():
    return CotAgent(
        agent_name="cot",
        system_prompt="Answer the question. Be concise.",  # deliberately weak — no abstain instruction
        output_type=Answer,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


def make_verifier():
    return VerifierAgent(
        agent_name="verifier",
        system_prompt=(
            "You receive a question and a candidate answer. Re-read the paragraph carefully, "
            "verify the answer is supported by the paragraph, and correct it if not. "
            "If the paragraph does not contain the answer, reply 'the paragraph does not say'."
        ),
        output_type=Answer,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


def make_abstainer():
    return CotAgent(
        agent_name="abstainer",
        system_prompt=(
            "For paragraphs that are off-topic or do not contain the answer, abstain by replying "
            "exactly 'the paragraph does not say'. Otherwise, answer the question concisely."
        ),
        output_type=Answer,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


agent_registry = {
    "cot": make_cot,
    "verifier": make_verifier,
    "abstainer": make_abstainer,
}
print(f"agent_registry: {sorted(agent_registry)}")


Task model:    openai:gpt-5.2
Reflection:    openai:gpt-5.2 (overridden via MetaAgentConfig if you want a stronger one)
agent_registry: ['abstainer', 'cot', 'verifier']


## 2. Gold set — factual / inferential / abstain (9 examples)

Three buckets the search will be measured against. Small set keeps cost low; the same shape as notebooks 06 / 07.

In [2]:
from orqest.optimization import GoldExample


def ex(question: str, paragraph: str, expected: str, bucket: str) -> GoldExample[str, Answer]:
    return GoldExample[str, Answer](
        input=f"Paragraph: {paragraph}\n\nQuestion: {question}",
        expected=Answer(answer=expected),
        id=bucket,
    )


# Factual — answer is in the paragraph literally
factual = [
    ex("In what year was Python first released?",
       "Python was first released in 1991 by Guido van Rossum.",
       "1991",
       "factual"),
    ex("Who is the CEO of Tesla?",
       "Tesla, headquartered in Austin, is led by CEO Elon Musk and produces electric vehicles.",
       "Elon Musk",
       "factual"),
    ex("What is the capital of Germany?",
       "Germany's capital, Berlin, is home to about 3.7 million people.",
       "Berlin",
       "factual"),
]

# Inferential — answer requires combining 2 facts
inferential = [
    ex("How many years between the release of Python and JavaScript?",
       "Python was released in 1991. JavaScript first appeared in 1995.",
       "4",
       "inferential"),
    ex("Which is older — the company or its CEO's tenure?",
       "Tesla was founded in 2003. Elon Musk became CEO in 2008.",
       "the company",
       "inferential"),
    ex("In which city does the older university operate?",
       "Harvard was founded in 1636 in Cambridge, Massachusetts. MIT was founded in 1861 also in Cambridge.",
       "Cambridge",
       "inferential"),
]

# Abstain — paragraph doesn't contain the answer; correct response is to abstain
abstain = [
    ex("What is the speed of light in km/s?",
       "This paragraph discusses the history of jazz music in New Orleans during the 1920s.",
       "the paragraph does not say",
       "abstain"),
    ex("Who painted the Mona Lisa?",
       "Solar panels convert sunlight into electricity at efficiencies typically between 15% and 22%.",
       "the paragraph does not say",
       "abstain"),
    ex("What is the population of Tokyo?",
       "Coffee originated in Ethiopia and was first cultivated in Yemen during the 15th century.",
       "the paragraph does not say",
       "abstain"),
]

gold_set = factual + inferential + abstain
print(f"Gold set: {len(gold_set)} examples ({len(factual)} factual / {len(inferential)} inferential / {len(abstain)} abstain)")


Gold set: 9 examples (3 factual / 3 inferential / 3 abstain)


## 3. Score function + per-bucket helper

In [3]:
def score_fn(output: Answer, example: GoldExample[str, Answer]) -> float:
    """Substring match: 1.0 if expected appears in output, else 0.0.

    Abstain bucket scores 1.0 only when the agent emits 'paragraph does not say'.
    """
    if example.expected is None:
        return 0.0
    out_low = output.answer.lower().strip()
    exp_low = example.expected.answer.lower().strip()
    if exp_low in out_low:
        return 1.0
    # For factual / inferential, also accept exact-equal trim
    if out_low == exp_low:
        return 1.0
    return 0.0


def per_bucket(bundles, examples):
    by_bucket: dict[str, list[float]] = {}
    for b, e in zip(bundles, examples):
        by_bucket.setdefault(e.id, []).append(b.accuracy)
    return {k: sum(v) / len(v) for k, v in by_bucket.items()}


def print_per_bucket(label, bucket_scores):
    print(f"{label}:")
    for bucket in ("factual", "inferential", "abstain"):
        score = bucket_scores.get(bucket)
        print(f"  {bucket:<14} {score:.2f}" if score is not None else f"  {bucket:<14} —")
    overall = sum(bucket_scores.values()) / len(bucket_scores) if bucket_scores else 0.0
    print(f"  {'overall':<14} {overall:.2f}")


## 4. Seed topology — single CoT pipeline

The W0 baseline. The meta agent will start from this and propose mutations.

In [4]:
from orqest.orchestration.spec import (
    AgentStepSpec, ParallelSpec, PipelineSpec, PipelineStepEntry, RouterSpec, RouteSpec,
)
from orqest.orchestration.hydrate import CallableRegistry

seed_topology = PipelineSpec(
    steps=[PipelineStepEntry(operation=AgentStepSpec(agent_name="cot"))]
)

# Callable registry — the meta agent can compose conditions / merges from these names.
def is_long(input_text):
    return len(str(input_text)) > 200

callable_registry = CallableRegistry()
callable_registry.register("is_long", is_long)
print(f"callable_registry: {callable_registry.names()}")
print(f"\nseed topology JSON:\n{seed_topology.model_dump_json(indent=2)}")


callable_registry: ['is_long']

seed topology JSON:
{
  "kind": "pipeline",
  "steps": [
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "cot",
        "inline_spec": null
      },
      "config": null
    }
  ],
  "name": "pipeline"
}


## 5. Baseline measurement

In [5]:
import asyncio

from orqest.optimization import TopologyEvaluator

evaluator = TopologyEvaluator(
    score_fn=score_fn,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    topology_gene_name="main",
)

baseline_bundles = await evaluator.evaluate_batch(
    decoded={"main": seed_topology},
    batch=gold_set,
)
baseline_buckets = per_bucket(baseline_bundles, gold_set)
print_per_bucket("BASELINE (single CoT)", baseline_buckets)


BASELINE (single CoT):
  factual        1.00
  inferential    1.00
  abstain        1.00
  overall        1.00


## 6. Run MetaAgentSearch (~3–5 min, ~\$1–2)

Conservative budget. `n_generations=5`, minibatch of 3 per candidate, `top_k` archive (per the [2510.06711 critique](https://arxiv.org/abs/2510.06711) — outperforms ADAS's original `cumulative` strategy). Each generation: design → reflexion → evaluate → archive.

In [6]:
from orqest.optimization import (
    MetaAgentConfig, MetaAgentSearch, TopologyGene,
)
from orqest.observability.events import EventBus

bus = EventBus()
events = []
bus.subscribe("meta_agent.iteration_completed", events.append)

gene = TopologyGene(
    name="main",
    initial=seed_topology,
    constraints=(
        "For abstain examples (off-topic paragraphs), the topology MUST end with an agent capable of abstaining ('the paragraph does not say'). "
        "Prefer simpler topologies — only add structure if it helps a specific bucket."
    ),
)

search = MetaAgentSearch(
    MetaAgentConfig(
        n_generations=5,
        archive_strategy="top_k",
        archive_size=3,
        reflexion_passes=1,   # 1 reflexion pass to keep cost down
        debug_max=2,
        minibatch_size=3,
        seed=42,
    ),
    gene=gene,
    evaluator=evaluator,
    meta_agent_model=config.llm_model,  # use your stronger model here in production
    api_key=config.llm_api_key,
    bus=bus,
)

result = await search.run(trainset=gold_set)
print(f"\nbest_score (aggregate, on minibatch): {result.best_score:.3f}")
print(f"archive size:                         {len(result.raw.entries)}")
print(f"pareto candidates:                    {len(result.pareto_candidates)}")



best_score (aggregate, on minibatch): 0.981
archive size:                         5
pareto candidates:                    1


## 7. Inspect archive — meta-agent's reasoning + Pareto front

In [7]:
print("=== Archive (newest → oldest) ===\n")
for entry in reversed(result.raw.entries):
    label = "SEED" if entry.generation == -1 else f"gen-{entry.generation}"
    print(f"--- {label}  score={entry.aggregate_score:.3f} ---")
    if entry.thought and entry.thought != "(seed)":
        thought = entry.thought.replace("\n", " ")
        print(f"  thought: {thought[:200]}")
    print()


print("=== Pareto front (accuracy ↑ / cost ↓ / latency ↓) ===\n")
for entry in result.raw.pareto():
    mean_acc = sum(b.accuracy for b in entry.bundles) / max(1, len(entry.bundles))
    mean_lat = sum(b.latency_ms for b in entry.bundles) / max(1, len(entry.bundles))
    n_agents = entry.bundles[0].raw.get("n_agents", 0) if entry.bundles else 0
    depth = entry.bundles[0].raw.get("depth", 0) if entry.bundles else 0
    label = "SEED" if entry.generation == -1 else f"gen-{entry.generation}"
    print(f"{label:>8}  acc={mean_acc:.2f}  lat={mean_lat:>6.0f}ms  n_agents={n_agents}  depth={depth}")


=== Archive (newest → oldest) ===

--- gen-4  score=0.970 ---
  thought: Hydration failed due to unknown callable state_updater_name. Callable registry only includes 'is_long', so avoid RefinementLoop or use 'is_long' where allowed. Provide a structurally different yet val

--- gen-2  score=0.620 ---
  thought: Hydration failed because Router had no matching route: the 'short' route had condition_name=null which is not treated as default match. Fix by adding fallback_step to router (or provide a condition pr

--- gen-1  score=0.981 ---
  thought: Hydration failed because state_updater_name referenced a non-existent callable. Callable registry only contains 'is_long'. Remove RefinementLoop (or set updater to an allowed callable). We'll replace 

--- gen-0  score=0.943 ---
  thought: Replace unknown state_updater_name with only known callable 'is_long' (acts as a placeholder updater) and simplify router by removing classifier to avoid route-name mismatch; keep structural change vs

--- S

## 8. Apply winner + measure per-bucket lift

`apply_result` swaps the topology in a registry dict (dry-run by default). The hydrated topology is then re-evaluated on the full gold set.

In [8]:
from orqest.optimization import apply_result

topology_registry = {"main": seed_topology}
diffs = apply_result(result, target=topology_registry, dry_run=True)
for d in diffs:
    if d.changed:
        print(f"would change {d.gene_name!r}:")
        print(d.unified[:1000])


would change 'main':
--- main (before)
+++ main (after)
@@ -3,12 +3,84 @@
   "steps": [
     {
       "operation": {
-        "kind": "agent_step",
-        "agent_name": "cot",
-        "inline_spec": null
+        "kind": "router",
+        "routes": [
+          {
+            "name": "complex",
+            "step": {
+              "kind": "pipeline",
+              "steps": [
+                {
+                  "operation": {
+                    "kind": "agent_step",
+                    "agent_name": "cot",
+                    "inline_spec": null
+                  },
+                  "config": null
+                },
+                {
+                  "operation": {
+                    "kind": "agent_step",
+                    "agent_name": "verifier",
+                    "inline_spec": null
+                  },
+                  "config": null
+                }
+              ],
+              "name": "complex_path"
+            },
+            "condition_name":

In [9]:
# Commit the swap, then re-evaluate against the full gold set
apply_result(result, target=topology_registry, dry_run=False)
winning = topology_registry["main"]
print(f"winning topology kind: {winning.kind}")

evolved_bundles = await evaluator.evaluate_batch(
    decoded={"main": winning},
    batch=gold_set,
)
evolved_buckets = per_bucket(evolved_bundles, gold_set)
print()
print_per_bucket("BASELINE", baseline_buckets)
print()
print_per_bucket("EVOLVED", evolved_buckets)
print()
print("Lift per bucket:")
for bucket in ("factual", "inferential", "abstain"):
    delta = evolved_buckets.get(bucket, 0) - baseline_buckets.get(bucket, 0)
    print(f"  {bucket:<14} {delta:+.2f}")


winning topology kind: pipeline



BASELINE:
  factual        1.00
  inferential    1.00
  abstain        1.00
  overall        1.00

EVOLVED:
  factual        1.00
  inferential    1.00
  abstain        1.00
  overall        1.00

Lift per bucket:
  factual        +0.00
  inferential    +0.00
  abstain        +0.00


## What's next

- **Notebook 09** combines this topology evolution with GEPA prompt evolution (two-phase: ADAS first, GEPA on the winner) and shows the full ablation.
- **W3.C — sandboxed codegen.** Once `orqest/sandbox/` ships, the meta agent can emit raw Python `forward()` functions (the original ADAS surface) for tasks where compositions of registered primitives are too constrained.
- **MCTS search.** AFlow-style MCTS over the same `TopologySpec` IR is a one-strategy swap inside `MetaAgentSearch` once the search-strategy abstraction lands.

For the design rationale (why TopologySpec instead of raw Python, why `top_k` archive, why two-phase composition with GEPA), see `docs/concepts/topology_optimization.md`.
